# My First ANN — MLP on Olist

Train a simple PyTorch neural network to predict whether an Olist review will be negative, using order-level features (delivery time, freight value, price, etc.).

**Setup:**
1. One hidden layer with ReLU
2. Output layer with a single logit (no sigmoid)
3. `BCEWithLogitsLoss` + `Adam`
4. Compare against Logistic Regression baseline

The last section walks through four common mistakes and shows what each one does to the model.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

device: cpu


## 2. Load Olist Data

Set `DATA_DIR` to the folder holding the CSVs. Default is `~/Downloads`; on Colab point it at `/content/`.

In [2]:
DATA_DIR = Path.home() / 'Downloads'

orders = pd.read_csv(
    DATA_DIR / 'olist_orders_dataset.csv',
    parse_dates=[
        'order_purchase_timestamp',
        'order_delivered_customer_date',
        'order_estimated_delivery_date',
    ],
)
reviews  = pd.read_csv(DATA_DIR / 'olist_order_reviews_dataset.csv')
items    = pd.read_csv(DATA_DIR / 'olist_order_items_dataset.csv')
payments = pd.read_csv(DATA_DIR / 'olist_order_payments_dataset.csv')

print('orders  :', orders.shape)
print('reviews :', reviews.shape)
print('items   :', items.shape)
print('payments:', payments.shape)

orders  : (99441, 8)
reviews : (99224, 7)
items   : (112650, 7)
payments: (103886, 5)


## 3. Feature Engineering

* **Target:** `is_negative` = 1 if `review_score ≤ 2`, else 0
* **Features:** `delivery_days`, `delivery_delay_days`, `freight_value`, `price`, `n_items`, `payment_installments`

In [3]:
orders = orders[orders['order_status'] == 'delivered'].copy()
orders['delivery_days'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.days
orders['delivery_delay_days'] = (
    orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
).dt.days

items_agg = items.groupby('order_id').agg(
    price=('price', 'sum'),
    freight_value=('freight_value', 'sum'),
    n_items=('order_item_id', 'count'),
).reset_index()

pay_agg = payments.groupby('order_id').agg(
    payment_value=('payment_value', 'sum'),
    payment_installments=('payment_installments', 'max'),
).reset_index()

rev_agg = reviews.groupby('order_id').agg(
    review_score=('review_score', 'mean'),
).reset_index()

df = (
    orders[['order_id', 'delivery_days', 'delivery_delay_days']]
    .merge(items_agg, on='order_id')
    .merge(pay_agg,   on='order_id')
    .merge(rev_agg,   on='order_id')
)
df['is_negative'] = (df['review_score'] <= 2).astype(int)
df = df.dropna()
df = df[(df['delivery_days'] >= 0) & (df['delivery_days'] <= 60)]

print(f'rows after cleanup: {len(df):,}')
df.head()

rows after cleanup: 95,550


,order_id,delivery_days,delivery_delay_days,price,freight_value,n_items,payment_value,payment_installments,review_score,is_negative
0,e481f51cbdc54678b7cc49136f2d6af7,8.0,-8.0,29.99,8.72,1,38.71,1,4.0,0
1,53cdb2fc8bc7dce0b6741e2150273451,13.0,-6.0,118.70,22.76,1,141.46,1,4.0,0
2,47770eb9100c2d0c44946d9cf07ec65d,9.0,-18.0,159.90,19.22,1,179.12,3,5.0,0
3,949d5b44dbf5de918fe9c16f97b45f8a,13.0,-13.0,45.00,27.20,1,72.20,1,5.0,0
4,ad21c59c0840e6cb83a9ceb5573f8159,2.0,-10.0,19.90,8.72,1,28.62,1,5.0,0


## 4. Class Balance

Olist reviews are imbalanced (~12–15% negative). Accuracy alone will be misleading — we also track precision, recall, F1, and ROC-AUC.

In [4]:
pos_rate = df['is_negative'].mean()
print(f'negative-review rate: {pos_rate:.1%}')
print(f'majority-class baseline accuracy: {1 - pos_rate:.3f}')

negative-review rate: 12.6%
majority-class baseline accuracy: 0.874


## 5. Train/Test Split and Scaling

`StandardScaler` is fit **on train only**. Fitting on the full dataset leaks test statistics into training.

`stratify=y` keeps the class ratio identical in both splits.

In [5]:
feature_cols = [
    'delivery_days', 'delivery_delay_days',
    'freight_value', 'price',
    'n_items', 'payment_installments',
]

X = df[feature_cols].values.astype(np.float32)
y = df['is_negative'].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y,
)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'X_train: {X_train_s.shape}')
print(f'X_test:  {X_test_s.shape}')
print(f'negative rate train={y_train.mean():.3f} test={y_test.mean():.3f}')

X_train: (76440, 6)
X_test:  (19110, 6)
negative rate train=0.126 test=0.126


## 6. Tensors and DataLoader

In [6]:
X_train_t = torch.tensor(X_train_s, dtype=torch.float32, device=device)
y_train_t = torch.tensor(y_train,   dtype=torch.float32, device=device)
X_test_t  = torch.tensor(X_test_s,  dtype=torch.float32, device=device)
y_test_t  = torch.tensor(y_test,    dtype=torch.float32, device=device)

BATCH_SIZE = 128
train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

print(f'batches per epoch: {len(train_loader):,}')

batches per epoch: 598


## 7. Define the MLP

6 inputs → 16 hidden with ReLU → 1 logit output. The final layer does **not** apply sigmoid; `BCEWithLogitsLoss` will do it internally in a numerically stable way.

In [7]:
class SimpleMLP(nn.Module):
    def __init__(self, n_features: int, hidden: int = 16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)

model = SimpleMLP(n_features=len(feature_cols), hidden=16).to(device)
print(model)

n_params = sum(p.numel() for p in model.parameters())
print(f'trainable parameters: {n_params:,}')

SimpleMLP(
  (net): Sequential(
    (0): Linear(in_features=6, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=1, bias=True)
  )
)
trainable parameters: 129


## 8. Training Loop

`pos_weight` in `BCEWithLogitsLoss` handles the class imbalance by up-weighting the positive class.

In [8]:
pos_rate = float(y_train.mean())
pos_weight = torch.tensor((1 - pos_rate) / pos_rate, device=device)
print(f'pos_weight = {pos_weight.item():.2f}')

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 15
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    avg = total_loss / len(train_ds)
    history.append(avg)
    if epoch == 1 or epoch % 3 == 0:
        print(f'epoch {epoch:2d}/{EPOCHS}  train_loss = {avg:.4f}')

pos_weight = 6.92
epoch  1/15  train_loss = 1.1001
epoch  3/15  train_loss = 1.0158
epoch  6/15  train_loss = 1.0082
epoch  9/15  train_loss = 1.0050
epoch 12/15  train_loss = 1.0038
epoch 15/15  train_loss = 1.0031


## 9. Evaluation

`model.eval()` disables layers like dropout; `torch.no_grad()` skips gradient tracking.

In [9]:
def evaluate(model, X_t, y_t, threshold=0.5):
    model.eval()
    with torch.no_grad():
        logits = model(X_t)
        probs  = torch.sigmoid(logits).cpu().numpy()
    y_true = y_t.cpu().numpy() if torch.is_tensor(y_t) else y_t
    y_pred = (probs >= threshold).astype(int)
    return {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
        'roc_auc':   roc_auc_score(y_true, probs),
        'y_pred':    y_pred,
        'probs':     probs,
    }

mlp_res = evaluate(model, X_test_t, y_test_t)
print('MLP')
for k in ('accuracy', 'precision', 'recall', 'f1', 'roc_auc'):
    print(f'  {k:10s}: {mlp_res[k]:.3f}')
print()
print(confusion_matrix(y_test, mlp_res['y_pred']))

MLP
  accuracy  : 0.805
  precision : 0.340
  recall    : 0.575
  f1        : 0.427
  roc_auc   : 0.756

[[14004  2695]
 [ 1024  1387]]


## 10. Logistic Regression Baseline

The interesting question: did the neural net actually beat logistic regression on this tabular data?

In [10]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)
lr.fit(X_train_s, y_train)

lr_probs = lr.predict_proba(X_test_s)[:, 1]
lr_pred  = (lr_probs >= 0.5).astype(int)

lr_res = {
    'accuracy':  accuracy_score(y_test, lr_pred),
    'precision': precision_score(y_test, lr_pred, zero_division=0),
    'recall':    recall_score(y_test, lr_pred, zero_division=0),
    'f1':        f1_score(y_test, lr_pred, zero_division=0),
    'roc_auc':   roc_auc_score(y_test, lr_probs),
}
print('Logistic Regression')
for k in ('accuracy', 'precision', 'recall', 'f1', 'roc_auc'):
    print(f'  {k:10s}: {lr_res[k]:.3f}')

Logistic Regression
  accuracy  : 0.750
  precision : 0.274
  recall    : 0.594
  f1        : 0.375
  roc_auc   : 0.738


In [11]:
cmp = pd.DataFrame({
    'MLP': {k: mlp_res[k] for k in ('accuracy','precision','recall','f1','roc_auc')},
    'LogisticRegression': lr_res,
})
cmp['diff (MLP - LR)'] = cmp['MLP'] - cmp['LogisticRegression']
cmp.round(3)

,MLP,LogisticRegression,diff (MLP - LR)
accuracy,0.805,0.750,0.056
precision,0.340,0.274,0.066
recall,0.575,0.594,-0.019
f1,0.427,0.375,0.053
roc_auc,0.756,0.738,0.018


## 11. Four Common Mistakes

Each subsection runs a broken variant so the failure mode is visible in numbers.

### Mistake 1 — Missing activation

Without a non-linearity between layers the network collapses into a single linear model. Its metrics end up nearly identical to logistic regression.

In [12]:
class LinearOnlyMLP(nn.Module):
    def __init__(self, n_features, hidden=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

bad = LinearOnlyMLP(len(feature_cols)).to(device)
opt = torch.optim.Adam(bad.parameters(), lr=1e-3)
crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

for _ in range(EPOCHS):
    bad.train()
    for xb, yb in train_loader:
        opt.zero_grad()
        crit(bad(xb), yb).backward()
        opt.step()

bad_res = evaluate(bad, X_test_t, y_test_t)
print(f'linear-only MLP:      F1={bad_res["f1"]:.3f}  ROC-AUC={bad_res["roc_auc"]:.3f}')
print(f'Logistic Regression:  F1={lr_res["f1"]:.3f}  ROC-AUC={lr_res["roc_auc"]:.3f}')

linear-only MLP:      F1=0.373  ROC-AUC=0.738
Logistic Regression:  F1=0.375  ROC-AUC=0.738


### Mistake 2 — Wrong learning rate

Too large → training diverges. Too small → training barely moves. `1e-3` is the standard first try with Adam.

In [13]:
for lr_test in [1e-1, 1e-3, 1e-6]:
    m2 = SimpleMLP(len(feature_cols)).to(device)
    opt = torch.optim.Adam(m2.parameters(), lr=lr_test)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    losses = []
    for _ in range(5):
        m2.train()
        epoch_loss = 0.0
        for xb, yb in train_loader:
            opt.zero_grad()
            loss = crit(m2(xb), yb)
            loss.backward()
            opt.step()
            epoch_loss += loss.item() * xb.size(0)
        losses.append(epoch_loss / len(train_ds))
    print(f'lr={lr_test:<8} losses = {[round(l, 3) for l in losses]}')

lr=0.1      losses = [1.039, 1.034, 1.034, 1.035, 1.035]
lr=0.001    losses = [1.069, 1.023, 1.017, 1.014, 1.012]
lr=1e-06    losses = [1.167, 1.167, 1.166, 1.165, 1.165]


### Mistake 3 — Skipping normalization

Features on wildly different scales (price in hundreds vs. items in single digits) make convergence unreliable. Compare the same architecture with and without `StandardScaler`.

In [14]:
X_train_raw_t = torch.tensor(X_train, dtype=torch.float32, device=device)
y_train_raw_t = torch.tensor(y_train, dtype=torch.float32, device=device)
X_test_raw_t  = torch.tensor(X_test,  dtype=torch.float32, device=device)

raw_loader = DataLoader(
    TensorDataset(X_train_raw_t, y_train_raw_t),
    batch_size=BATCH_SIZE, shuffle=True,
)

m3 = SimpleMLP(len(feature_cols)).to(device)
opt = torch.optim.Adam(m3.parameters(), lr=1e-3)
crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

for _ in range(EPOCHS):
    m3.train()
    for xb, yb in raw_loader:
        opt.zero_grad()
        crit(m3(xb), yb).backward()
        opt.step()

raw_res = evaluate(m3, X_test_raw_t, torch.tensor(y_test, dtype=torch.float32, device=device))
print(f'no scaler:   F1={raw_res["f1"]:.3f}  ROC-AUC={raw_res["roc_auc"]:.3f}')
print(f'with scaler: F1={mlp_res["f1"]:.3f}  ROC-AUC={mlp_res["roc_auc"]:.3f}')

no scaler:   F1=0.420  ROC-AUC=0.741
with scaler: F1=0.427  ROC-AUC=0.756


### Mistake 4 — Confusing logits and probabilities

Adding `nn.Sigmoid()` before `BCEWithLogitsLoss` applies sigmoid twice — a silent bug that trains without errors but produces poor gradients.

In [15]:
class BuggyMLP(nn.Module):
    def __init__(self, n_features, hidden=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

buggy = BuggyMLP(len(feature_cols)).to(device)
opt = torch.optim.Adam(buggy.parameters(), lr=1e-3)
crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

for _ in range(EPOCHS):
    buggy.train()
    for xb, yb in train_loader:
        opt.zero_grad()
        crit(buggy(xb), yb).backward()
        opt.step()

buggy_res = evaluate(buggy, X_test_t, y_test_t)
print(f'double-sigmoid MLP: F1={buggy_res["f1"]:.3f}  ROC-AUC={buggy_res["roc_auc"]:.3f}')
print(f'correct MLP:        F1={mlp_res["f1"]:.3f}  ROC-AUC={mlp_res["roc_auc"]:.3f}')

double-sigmoid MLP: F1=0.224  ROC-AUC=0.737
correct MLP:        F1=0.427  ROC-AUC=0.756


## 12. Summary

| Question | Answer |
|----------|--------|
| Did the MLP beat logistic regression? | Usually barely, sometimes not at all. |
| Why? | Six mostly-linear tabular features. Linear models handle this cleanly. |
| When does deep learning pay off? | Unstructured data (images, audio, raw text) or tabular data with strong interactions and lots of rows. |
| What did the exercise teach? | The five-step PyTorch training loop and how to recognize the four common bugs when they happen for real. |

## 13. Regularization: Dropout + Weight Decay

Train four variants of the same MLP and compare the train/validation gap. A big gap is overfitting; the goal is to shrink it without losing validation performance.

1. **baseline** - no regularization
2. **dropout** only (p=0.3)
3. **weight_decay** only (AdamW, wd=0.01)
4. **both** combined

A deeper network (`6 -> 32 -> 32 -> 1`) is used here to make the overfitting visible in a small dataset.

In [ ]:
from sklearn.model_selection import train_test_split as tts
import matplotlib.pyplot as plt

X_tr_np, X_val_np, y_tr_np, y_val_np = tts(
    X_train_s, y_train, test_size=0.1, random_state=SEED, stratify=y_train,
)

X_tr_t  = torch.tensor(X_tr_np,  dtype=torch.float32, device=device)
y_tr_t  = torch.tensor(y_tr_np,  dtype=torch.float32, device=device)
X_val_t = torch.tensor(X_val_np, dtype=torch.float32, device=device)
y_val_t = torch.tensor(y_val_np, dtype=torch.float32, device=device)

tr_ds = TensorDataset(X_tr_t, y_tr_t)
tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True)

pos_rate_tr = float(y_tr_np.mean())
pw = torch.tensor((1 - pos_rate_tr) / pos_rate_tr, device=device)
print(f'train={len(X_tr_np):,}  val={len(X_val_np):,}  pos_weight={pw.item():.2f}')

In [ ]:
class RegMLP(nn.Module):
    def __init__(self, n_features, hidden=32, dropout=0.0):
        super().__init__()
        layers = [nn.Linear(n_features, hidden), nn.ReLU()]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        layers += [nn.Linear(hidden, hidden), nn.ReLU()]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        layers.append(nn.Linear(hidden, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_with_tracking(model, epochs, lr=1e-3, weight_decay=0.0):
    if weight_decay > 0:
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    train_losses, val_losses = [], []
    for _ in range(epochs):
        model.train()
        total = 0.0
        for xb, yb in tr_loader:
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
            total += loss.item() * xb.size(0)
        train_losses.append(total / len(X_tr_np))
        model.eval()
        with torch.no_grad():
            val_losses.append(crit(model(X_val_t), y_val_t).item())
    return train_losses, val_losses

In [ ]:
REG_EPOCHS = 30

configs = [
    ('baseline',       0.0, 0.0),
    ('dropout',        0.3, 0.0),
    ('weight_decay',   0.0, 0.01),
    ('dropout + wd',   0.3, 0.01),
]

variants = {}
for name, dp, wd in configs:
    torch.manual_seed(SEED)
    m = RegMLP(len(feature_cols), hidden=32, dropout=dp).to(device)
    tr_l, va_l = train_with_tracking(m, REG_EPOCHS, weight_decay=wd)
    variants[name] = {'model': m, 'train': tr_l, 'val': va_l}
    print(f'{name:16s}  train={tr_l[-1]:.4f}  val={va_l[-1]:.4f}  gap={va_l[-1]-tr_l[-1]:+.4f}')

In [ ]:
gap_summary = pd.DataFrame({
    name: {
        'final_train_loss': d['train'][-1],
        'final_val_loss':   d['val'][-1],
        'gap (val - train)': d['val'][-1] - d['train'][-1],
        'best_val_loss':    min(d['val']),
        'best_epoch':       int(np.argmin(d['val'])) + 1,
    }
    for name, d in variants.items()
}).T.round(4)
gap_summary

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
for ax, (name, d) in zip(axes, variants.items()):
    ax.plot(d['train'], label='train', color='#0d9488')
    ax.plot(d['val'],   label='val',   color='#dc2626')
    ax.set_title(name)
    ax.set_xlabel('epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('loss')
plt.tight_layout()
plt.show()

## 14. Early Stopping

Take the overfitting baseline and add early stopping with `patience=7`. Measure how many epochs were saved vs. a fixed 50-epoch budget.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=7, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.best_epoch = 0
        self.best_state = None

    def check(self, val_loss, model, epoch):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_epoch = epoch
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
            self.counter = 0
            return False
        self.counter += 1
        return self.counter >= self.patience

In [ ]:
FULL_EPOCHS = 50

torch.manual_seed(SEED)
full_model = RegMLP(len(feature_cols), hidden=32, dropout=0.0).to(device)
full_tr, full_va = train_with_tracking(full_model, FULL_EPOCHS)
print(f'Full run: {FULL_EPOCHS} epochs, best val loss = {min(full_va):.4f} at epoch {int(np.argmin(full_va))+1}')

In [ ]:
torch.manual_seed(SEED)
es_model = RegMLP(len(feature_cols), hidden=32, dropout=0.0).to(device)
es_opt = torch.optim.Adam(es_model.parameters(), lr=1e-3)
es_crit = nn.BCEWithLogitsLoss(pos_weight=pw)
es = EarlyStopping(patience=7)

es_tr, es_va = [], []
epochs_run = 0
for ep in range(1, FULL_EPOCHS + 1):
    es_model.train()
    total = 0.0
    for xb, yb in tr_loader:
        es_opt.zero_grad()
        loss = es_crit(es_model(xb), yb)
        loss.backward()
        es_opt.step()
        total += loss.item() * xb.size(0)
    es_tr.append(total / len(X_tr_np))
    es_model.eval()
    with torch.no_grad():
        v = es_crit(es_model(X_val_t), y_val_t).item()
    es_va.append(v)
    epochs_run = ep
    if es.check(v, es_model, ep):
        break

es_model.load_state_dict(es.best_state)
print(f'With EarlyStopping: {epochs_run} epochs, best val loss = {es.best_loss:.4f} at epoch {es.best_epoch}')
print(f'Epochs saved: {FULL_EPOCHS - epochs_run} ({(FULL_EPOCHS - epochs_run) / FULL_EPOCHS:.0%} of budget)')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, FULL_EPOCHS + 1), full_va, label='full run val loss', color='#94a3b8', alpha=0.7)
ax.plot(range(1, epochs_run + 1),  es_va,   label='early-stop val loss', color='#0d9488', linewidth=2)
ax.axvline(es.best_epoch, color='#16a34a', linestyle='--', label=f'best epoch = {es.best_epoch}')
ax.axvline(epochs_run,    color='#dc2626', linestyle=':',  label=f'stopped at = {epochs_run}')
ax.set_xlabel('epoch')
ax.set_ylabel('val loss')
ax.set_title('Early stopping saves compute without hurting best val loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 15. Autoencoder for Anomaly Detection

Train a fully unsupervised autoencoder on the six order features. Orders the autoencoder cannot reconstruct well are our anomaly candidates. Architecture: `6 -> 4 -> 2 -> 4 -> 6`. The 2-neuron bottleneck forces the network to keep only the most compressible structure of a typical Olist order.

Note: we fit a fresh scaler on the union of train+test since anomaly detection is unsupervised - no labels involved.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, n_features=6, bottleneck=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 4), nn.ReLU(),
            nn.Linear(4, bottleneck),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, 4), nn.ReLU(),
            nn.Linear(4, n_features),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

In [ ]:
X_all_np = np.vstack([X_train, X_test])
scaler_all = StandardScaler().fit(X_all_np)
X_all_s = scaler_all.transform(X_all_np)
X_all_t = torch.tensor(X_all_s, dtype=torch.float32, device=device)

ae_loader = DataLoader(TensorDataset(X_all_t), batch_size=256, shuffle=True)

torch.manual_seed(SEED)
ae = Autoencoder(n_features=len(feature_cols), bottleneck=2).to(device)
ae_opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
ae_crit = nn.MSELoss()

AE_EPOCHS = 30
ae_losses = []
for ep in range(1, AE_EPOCHS + 1):
    ae.train()
    total = 0.0
    for (xb,) in ae_loader:
        ae_opt.zero_grad()
        loss = ae_crit(ae(xb), xb)
        loss.backward()
        ae_opt.step()
        total += loss.item() * xb.size(0)
    ae_losses.append(total / len(X_all_t))
    if ep == 1 or ep % 5 == 0:
        print(f'epoch {ep:2d}/{AE_EPOCHS}  reconstruction MSE = {ae_losses[-1]:.4f}')

In [ ]:
ae.eval()
with torch.no_grad():
    recon = ae(X_all_t)
    per_sample_error = ((X_all_t - recon) ** 2).mean(dim=1).cpu().numpy()

df_all = pd.DataFrame(X_all_np, columns=feature_cols)
df_all['reconstruction_error'] = per_sample_error

threshold = np.quantile(per_sample_error, 0.99)
df_all['is_anomaly'] = (df_all['reconstruction_error'] >= threshold).astype(int)

n_anomalies = df_all['is_anomaly'].sum()
print(f'99th-percentile threshold: {threshold:.4f}')
print(f'orders flagged as anomalies: {n_anomalies:,} ({n_anomalies / len(df_all):.2%})')

In [ ]:
df_all.nlargest(10, 'reconstruction_error').round(2)

In [ ]:
normal_stats = df_all[df_all['is_anomaly'] == 0][feature_cols].describe().T[['mean', '50%', 'max']]
anomaly_stats = df_all[df_all['is_anomaly'] == 1][feature_cols].describe().T[['mean', '50%', 'max']]

pd.concat({'normal': normal_stats, 'anomalous': anomaly_stats}, axis=1).round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(per_sample_error, bins=100, color='#0d9488', alpha=0.7)
ax.axvline(threshold, color='#dc2626', linestyle='--', label=f'top 1% cutoff = {threshold:.3f}')
ax.set_xlabel('reconstruction error (MSE per order)')
ax.set_ylabel('count (log)')
ax.set_title('Reconstruction error distribution - the tail is the anomaly zone')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 16. Discussion

Two questions the exercise poses.

**1. Did regularization actually reduce overfitting?**

The `gap (val - train)` column from section 13 is the direct answer. A smaller gap means the network generalized better. If the gap shrank while the best val loss stayed low, regularization did its job. If the gap shrank but the val loss got worse, the model is now underfitting - regularization was too strong.

**2. Do the autoencoder anomalies look suspicious?**

The top-10 table above lists the ten orders the autoencoder failed hardest to reconstruct. Compare them to the `normal` vs. `anomalous` summary table. Common patterns you should see: very late deliveries (delivery_days near 60), unusually large basket size, extreme freight relative to price, or many installments. Those combinations rarely appear together in typical Olist orders, which is why the compressed 2-neuron representation cannot capture them.

This is the data-scientist interpretation step - the model produces candidates, you decide if they mean fraud, entry error, extreme customers, or noise. The model does not know.

In [ ]:
print('Regularization impact on train/val gap:')
print(gap_summary[['final_train_loss', 'final_val_loss', 'gap (val - train)']])

In [ ]:
top5 = df_all.nlargest(5, 'reconstruction_error').copy()
avg_features = df_all[feature_cols].mean()

for i, (_, row) in enumerate(top5.iterrows(), 1):
    reasons = []
    for col in feature_cols:
        avg = avg_features[col]
        val = row[col]
        if abs(avg) > 0.01 and val > 4 * avg:
            reasons.append(f'{col}={val:.1f} (avg {avg:.1f})')
    tag = '; '.join(reasons) if reasons else 'unusual combination'
    print(f"#{i}  error={row['reconstruction_error']:.3f}  ->  {tag}")